# NERISAL: INT8 quantisation for Arm (Google Colab)

Makes INT8 versions of a YOLO `.pt` model for ExecuTorch (XNNPACK), then compares **FP32 vs INT8**:
same images, same post-processing, people/head counts side by side.

**Design choice:** everything is INT8 **except the final Detect layer**. That layer outputs box coordinates
(0–640) and class scores (0–1) in one tensor; quantising it would crush the scores. Keeping it FP32 is a
standard mixed-precision approach.

**Today (dry run):** `yolov8n.pt` (pretrained, person class).
**Friday:** set `WEIGHTS` to `head640/best.pt` and `head320/best.pt` and `TARGET_CLASS = 0`, run again.

Runtime: CPU is fine. Calibration photos (220 real crowd images from CrowdHuman) are downloaded automatically.

In [ ]:
# 1. Install (ExecuTorch pinned to the Pi's version)
!pip install -q ultralytics executorch==1.6.0
!pip show executorch | grep Version

In [ ]:
# 2. Settings
WEIGHTS = {"yolov8n_640": ("yolov8n.pt", 640), "yolov8n_320": ("yolov8n.pt", 320)}
# Friday: WEIGHTS = {"head640": ("models/head640/best.pt", 640), "head320": ("models/head320/best.pt", 320)}
TARGET_CLASS = 0          # COCO person = 0; head models also use 0
CONF = 0.25
N_CALIB = 120             # calibration images (more = slower, usually slightly better)

In [ ]:
# 3. Calibration / test images: downloaded automatically (220 CrowdHuman crowd photos), no manual work.
#    (If you prefer your own photos, put them in calib.zip in the file panel before running; it will be used instead.)
import os, glob, zipfile, random
os.makedirs("calib", exist_ok=True)
if os.path.exists("calib.zip"):
    zipfile.ZipFile("calib.zip").extractall("calib")
else:
    !pip install -q huggingface_hub
    from huggingface_hub import list_repo_files, hf_hub_download
    REPO = "ShantyCam/crowdhuman"
    f = [x for x in list_repo_files(REPO, repo_type="dataset") if x.endswith("CrowdHuman_val.zip")][0]
    print("downloading", f, "(2.5 GB, about 1-3 min on Colab)")
    zp = hf_hub_download(REPO, f, repo_type="dataset", local_dir="ch")
    with zipfile.ZipFile(zp) as zf:
        jpgs = sorted(n for n in zf.namelist() if n.lower().endswith(".jpg"))
        for n in random.Random(0).sample(jpgs, 220):          # extract only the 220 we need
            zf.extract(n, "calib")
    os.remove(zp)                                              # free the disk space
imgs = sorted(f for e in ("jpg", "jpeg", "png", "webp") for f in glob.glob(f"calib/**/*.{e}", recursive=True))
random.Random(0).shuffle(imgs)
test_imgs, calib_imgs = imgs[:20], imgs[20:20 + N_CALIB] or imgs[:N_CALIB]
print(len(imgs), "images:", len(calib_imgs), "for calibration,", len(test_imgs), "for the FP32 vs INT8 test")

In [ ]:
# 4. Helpers: letterbox exactly like nerisal.py, and count detections from a raw (1, 4+C, N) output
import cv2, numpy as np, torch, torchvision

def letterbox(img, size):
    h, w = img.shape[:2]; r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    c = np.full((size, size, 3), 114, np.uint8); t, l = (size - nh) // 2, (size - nw) // 2
    c[t:t + nh, l:l + nw] = cv2.resize(img, (nw, nh))
    x = torch.from_numpy(np.ascontiguousarray(c[:, :, ::-1].transpose(2, 0, 1))).float().unsqueeze(0) / 255
    return x

def count(out, conf=CONF, cls=TARGET_CLASS):
    p = out[0].T                                     # (N, 4 + C)
    s = p[:, 4 + cls]; k = s > conf
    if k.sum() == 0: return 0
    b, s = p[k, :4], s[k]
    xyxy = torch.cat([b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2], 1)
    return int(len(torchvision.ops.nms(xyxy, s, 0.5)))

In [ ]:
# 5. Build the exportable model (fused, Detect head in export mode, anchors cached by a dry run)
from ultralytics import YOLO

def export_model(weights, size):
    m = YOLO(weights).model.float().eval()
    m = m.fuse()                                   # fold BatchNorm into conv (better for INT8)
    for mod in m.modules():
        if hasattr(mod, "export"): mod.export = True
        if hasattr(mod, "format"): mod.format = "executorch"
    with torch.no_grad():
        m(torch.zeros(1, 3, size, size))           # caches anchors / strides for this input size
    return m

# quantisation API moved between versions: try the new location first
try:
    from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
except ImportError:
    from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
print("imports OK")

In [ ]:
# 6. Export FP32 and INT8 .pte for each model
import time
os.makedirs("pte", exist_ok=True)
built = {}
for name, (w, size) in WEIGHTS.items():
    m = export_model(w, size); x = torch.zeros(1, 3, size, size)
    with torch.no_grad():
        ref = m(x)
    print(f"{name}: output {tuple(ref.shape)}")

    # FP32
    ep = torch.export.export(m, (x,))
    open(f"pte/{name}_fp32.pte", "wb").write(
        to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch().buffer)

    # INT8 (per-channel weights, everything except the final Detect layer)
    q = XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True))
    detect_type = type(m.model[-1])                 # the Detect layer class
    try:
        q.set_module_type(detect_type, None)        # None = do not quantise this layer type: keep FP32
        print("  Detect layer kept FP32:", detect_type.__name__)
    except Exception as e:
        print("  WARNING: could not keep Detect in FP32, whole model INT8 (scores may suffer):", e)
    gm = torch.export.export(m, (x,)).module()
    gm = prepare_pt2e(gm, q)
    t = time.time()
    with torch.no_grad():
        for p in calib_imgs:
            im = cv2.imread(p)
            if im is not None: gm(letterbox(im, size))
    gm = convert_pt2e(gm)
    print(f"  calibrated on {len(calib_imgs)} images in {time.time() - t:.0f}s")
    ep8 = torch.export.export(gm, (x,))
    open(f"pte/{name}_int8.pte", "wb").write(
        to_edge_transform_and_lower(ep8, partitioner=[XnnpackPartitioner()]).to_executorch().buffer)
    built[name] = size
    for k in ("fp32", "int8"):
        f = f"pte/{name}_{k}.pte"; print(f"  {f}: {os.path.getsize(f)/1e6:.1f} MB")

In [ ]:
# 7. FP32 vs INT8: counts on the same test images (the number that matters), plus Colab CPU time
from executorch.runtime import Runtime
for name, size in built.items():
    meths = {k: Runtime.get().load_program(f"pte/{name}_{k}.pte").load_method("forward") for k in ("fp32", "int8")}
    diffs, rows = [], []
    t = {"fp32": [], "int8": []}
    for p in test_imgs:
        im = cv2.imread(p)
        if im is None: continue
        x = letterbox(im, size); c = {}
        for k, m in meths.items():
            t0 = time.perf_counter(); out = m.execute([x])[0]; t[k].append((time.perf_counter() - t0) * 1000)
            c[k] = count(out)
        rows.append((os.path.basename(p)[:18], c["fp32"], c["int8"]))
        diffs.append(abs(c["int8"] - c["fp32"]) / max(c["fp32"], 1) * 100)
    print(f"\n{name}: image, FP32 count, INT8 count")
    for r in rows: print(f"  {r[0]:20s}{r[1]:6d}{r[2]:6d}")
    print(f"  mean count difference: {np.mean(diffs):.1f}%   "
          f"Colab CPU ms: FP32 {np.mean(t['fp32']):.0f}, INT8 {np.mean(t['int8']):.0f} (the Pi decides the real speed)")

In [ ]:
# 8. Zip for the Builder (B times FP32 vs INT8 on the Pi: that is the speed-ladder row)
with zipfile.ZipFile("nerisal_int8_pte.zip", "w") as z:
    for f in glob.glob("pte/*.pte"): z.write(f, os.path.basename(f))
print(zipfile.ZipFile("nerisal_int8_pte.zip").namelist())
from google.colab import files; files.download("nerisal_int8_pte.zip")